In [3]:
from pyspark.sql import SparkSession
import csv
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

BRONZE_DIR = "/lakehouse/default/Files/bronze"

# helper functions

def read_csv(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def missing(value):
    return value is None or str(value).strip() == "" or str(value).strip().lower() == "null"

def to_int(value, default=None):
    try:
        return int(float(str(value).strip()))
    except:
        return default

def to_float(value, default=None):
    try:
        return float(str(value).strip())
    except:
        return default

def parse_date(value):
    if missing(value):
        return None

    value = str(value).strip()
    formats = ["%Y-%m-%d", "%d.%m.%Y", "%d/%m/%Y"]

    for fmt in formats:
        try:
            return datetime.strptime(value, fmt).strftime("%Y-%m-%d")
        except:
            pass

    return None

def clean_text(value, default="unknown"):
    return default if missing(value) else str(value).strip()

def clean_status(value):
    if missing(value):
        return None
    value = str(value).strip().lower()
    if value in ["completed", "shipped", "cancelled"]:
        return value
    return None

def clean_customer_name(value):
    if missing(value):
        return "unknown"
    parts = str(value).strip().split()
    parts.sort()
    return " ".join(parts)

def deduplicate(rows, key):
    seen = set()
    result = []
    for row in rows:
        if row[key] not in seen:
            seen.add(row[key])
            result.append(row)
    return result

def save_table(rows, table_name):
    df = spark.createDataFrame(rows)
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)

def impute_region_from_title(store_title, current_region):
    if not missing(current_region):
        return str(current_region).strip().lower()

    if missing(store_title):
        return "unknown"

    words = str(store_title).strip().split()

    for i, word in enumerate(words):
        if word.lower() == "store" and i + 1 < len(words):
            return words[i + 1].strip().lower()

    return "unknown"

# cleaning / transformation

def clean_customers():
    rows = read_csv(f"{BRONZE_DIR}/customers.csv")
    cleaned = []

    for row in rows:
        customer_id = to_int(row["CustomerID"])
        if customer_id is None:
            continue

        cleaned.append({
            "customer_id": customer_id,
            "customer_name": clean_customer_name(row["Name"]),
            "city": clean_text(row["City"]),
            "registration_date": parse_date(row["Registration Date"]),
            "customer_type": clean_text(row["Type"])
        })

    cleaned = deduplicate(cleaned, "customer_id")
    save_table(cleaned, "slv_customers")
    print("slv_customers created")

def clean_products():
    rows = read_csv(f"{BRONZE_DIR}/products.csv")
    cleaned = []

    for row in rows:
        product_id = to_int(row["Product"])
        if product_id is None:
            continue

        cleaned.append({
            "product_id": product_id,
            "product_title": clean_text(row["Title"]),
            "category": clean_text(row["Category"]),
            "cost": to_float(row["Cost"], None)
        })

    cleaned = deduplicate(cleaned, "product_id")
    save_table(cleaned, "slv_products")
    print("slv_products created")

def clean_stores():
    rows = read_csv(f"{BRONZE_DIR}/stores.csv")
    cleaned = []

    for row in rows:
        store_id = to_int(row["Store"])
        if store_id is None or store_id == 0:
            continue

        store_title = clean_text(row["Title"])

        cleaned.append({
            "store_id": store_id,
            "store_title": store_title,
            "city": clean_text(row["City"]),
            "region": impute_region_from_title(store_title, row["Region"])
        })

    cleaned = deduplicate(cleaned, "store_id")
    save_table(cleaned, "slv_stores")
    print("slv_stores created")

def clean_orders():
    rows = read_csv(f"{BRONZE_DIR}/orders.csv")
    cleaned = []
    dropped_store = 0
    dropped_status = 0

    for row in rows:
        store_id = to_int(row["Store"], 0)
        order_status = clean_status(row["Status"])
        order_id = to_int(row["Order"])

        if order_id is None:
            continue

        if store_id == 0:
            dropped_store += 1
            continue

        if order_status == None:
            dropped_status += 1
            continue

        cleaned.append({
            "order_id": order_id,
            "customer_name": clean_customer_name(row["Customer Name"]),
            "store_id": store_id,
            "order_date": parse_date(row["Date"]),
            "order_status": order_status
        })

    cleaned = deduplicate(cleaned, "order_id")
    save_table(cleaned, "slv_orders")
    print("slv_orders created")
    print(f"Dropped orders with store_id = 0: {dropped_store}")
    print(f"Dropped orders with unknown status: {dropped_status}")

def clean_order_items():
    rows = read_csv(f"{BRONZE_DIR}/order_items.csv")
    cleaned = []
    dropped_qty = 0

    for row in rows:
        item_id = to_int(row["Item"])
        quantity = to_int(row["Qty"], None)

        if item_id is None:
            continue

        if quantity is None or quantity <= 0:
            dropped_qty += 1
            continue

        cleaned.append({
            "item_id": item_id,
            "order_id": to_int(row["Order"]),
            "product_title": clean_text(row["Product"]),
            "quantity": quantity,
            "unit_price": to_float(row["Price"])
        })

    cleaned = deduplicate(cleaned, "item_id")
    save_table(cleaned, "slv_order_items")
    print("slv_order_items created")
    print(f"Dropped rows with invalid quantity: {dropped_qty}")

# run

clean_customers()
clean_products()
clean_stores()
clean_orders()
clean_order_items()

print("Silver Layer (Delta Tables) completed")

StatementMeta(, ef391e92-8d9f-4dfd-bdf3-9b956bedb7a8, 5, Finished, Available, Finished, False)

slv_customers created
slv_products created
slv_stores created
slv_orders created
Dropped orders with store_id = 0: 13
Dropped orders with unknown status: 7
slv_order_items created
Dropped rows with invalid quantity: 26
Silver Layer (Delta Tables) completed


In [4]:
df = spark.sql("SELECT * FROM sales_lakehouse.slv_customers LIMIT 1000")
display(df)

StatementMeta(, ef391e92-8d9f-4dfd-bdf3-9b956bedb7a8, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4a61b006-c5a2-406e-9200-9551c67ee32d)

In [5]:
df = spark.sql("SELECT * FROM sales_lakehouse.slv_order_items LIMIT 1000")
display(df)

StatementMeta(, ef391e92-8d9f-4dfd-bdf3-9b956bedb7a8, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72d077ec-bc77-4561-a604-8c5db31d9d10)

In [6]:
df = spark.sql("SELECT * FROM sales_lakehouse.slv_orders LIMIT 1000")
display(df)

StatementMeta(, ef391e92-8d9f-4dfd-bdf3-9b956bedb7a8, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 69915921-a520-4523-a238-d058646d5d56)

In [7]:
df = spark.sql("SELECT * FROM sales_lakehouse.slv_products LIMIT 1000")
display(df)

StatementMeta(, ef391e92-8d9f-4dfd-bdf3-9b956bedb7a8, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9a9e68c2-0ebd-4c6d-9a7b-ecee2f57ad09)

In [8]:
df = spark.sql("SELECT * FROM sales_lakehouse.slv_stores LIMIT 1000")
display(df)

StatementMeta(, ef391e92-8d9f-4dfd-bdf3-9b956bedb7a8, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 630de28f-12eb-4faf-a229-ab45504f1b41)